# Project 03: Convolution, a Small CNN, and Sequence Models

Build a convolution and max pooling step by hand with numpy, then train a small CNN on handwritten digit images with PyTorch.

Run top to bottom. Each step matches the README.

In [ ]:
# Step 0: Set up the project folder (Colab)
import os
import sys

PROJECT = "projects/03-cnn-and-sequence-models"

if "google.colab" in sys.modules:
    os.chdir("/content")
    if os.path.isdir(os.path.join("/content", PROJECT)):
        os.chdir(os.path.join("/content", PROJECT))
    else:
        print("Project folder not found. Upload the folder 03-cnn-and-sequence-models into /content/projects/ (folder icon in the left sidebar), then run this cell again.")
print("Working folder:", os.getcwd())

## Step 1: install packages

Colab already includes PyTorch (a CUDA build). The local `requirements.txt` pins `torch==2.6.0+cpu`, which is a CPU-only wheel that will NOT install on Colab. So this notebook does not install torch. The cell below installs only the numpy and scikit-learn pins from `requirements.txt`, and keeps the torch that Colab already has.

In [ ]:
%pip install -q numpy==2.1.3 scikit-learn==1.6.1

## Step 2: API key

This project needs **no** API key.

## Part A: numpy convolution and pooling (`cnn_steps.py`)

### README Step 1: Read the image as numbers
**What:** A grayscale image is a grid of pixels. Each pixel is a number from 0 (black) to 255 (white).

**Why:** A CNN never sees a picture. It sees a matrix of numbers.

In [ ]:
import numpy as np

# Step 1: A 5x5 grayscale image. Each number is a pixel from 0 (black) to 255 (white).
pixels = np.array([
    [10, 40, 200, 220, 30],
    [15, 50, 210, 230, 25],
    [12, 45, 205, 225, 35],
    [14, 42, 198, 218, 28],
    [11, 38, 202, 215, 32],
])
print("Step 1: Pixel values (0 to 255)")
print(pixels)

### README Step 2: Normalize
**What:** Divide every pixel by 255 so all values are between 0 and 1.

**Why:** Small numbers keep the arithmetic simple and stable. This is the step the course calls normalization.

In [ ]:
# Step 2: Normalize so every value is between 0 and 1.
feature_map = pixels / 255.0
print("\nStep 2: Normalized feature map (0 to 1), rounded to 2 decimals")
print(np.round(feature_map, 2))

### README Step 3: Slide a filter (convolution)
**What:** A 3x3 filter (also called a kernel) moves across the 5x5 feature map. At each position, multiply the overlapping values and add them up. That one sum is one value in the output.

**Why:** The filter is a small pattern detector. This filter responds to vertical edges. A negative output means the right side of the patch is brighter than the left.

The function `convolve2d` is defined in this cell, with the filter.

In [ ]:
def convolve2d(feature_map, kernel):
    """Slide the kernel over the feature map (stride 1, no padding).

    At each position, multiply the overlapping values and add them up.
    Deep-learning libraries call this operation "convolution", although
    strictly it is cross-correlation (the kernel is not flipped).
    """
    n = feature_map.shape[0]   # N: input size
    f = kernel.shape[0]        # F: filter size
    out_size = n - f + 1       # Output = N - F + 1
    output = np.zeros((out_size, out_size))

    for row in range(out_size):
        for col in range(out_size):
            patch = feature_map[row:row + f, col:col + f]
            output[row, col] = np.sum(patch * kernel)
    return output


# Step 3: A 3x3 filter (kernel). This one responds to vertical edges.
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
])
print("\nStep 3: 3x3 filter")
print(kernel)

### README Step 4: Check the output size
**What:** The output size is `N - F + 1`. Here N = 5 and F = 3, so the output is 3x3.

**Why:** You can predict the shape of any convolution before running it. This is the size rule from the 2026-10-28 session.

In [ ]:
# Step 4: Convolution and the output size check.
conv_out = convolve2d(feature_map, kernel)
expected_size = feature_map.shape[0] - kernel.shape[0] + 1
print("\nStep 4: Convolution output (3x3), rounded to 2 decimals")
print(np.round(conv_out, 2))
print(f"Output size check: N - F + 1 = {feature_map.shape[0]} - {kernel.shape[0]} + 1 = {expected_size}")
assert conv_out.shape == (expected_size, expected_size), "Output size does not match N - F + 1"
print(f"Check passed: output is {conv_out.shape[0]}x{conv_out.shape[1]}, from {feature_map.size} values down to {conv_out.size}")

### README Step 5: Max pooling
**What:** A 2x2 window slides over the 3x3 output. Each window keeps only its largest value. The result is 2x2.

**Why:** Pooling shrinks the feature map and keeps the strongest signal. This is the pooling step from the 2026-11-02 session.

The function `max_pool2d` is defined in this cell.

In [ ]:
def max_pool2d(feature_map, window=2):
    """Slide a window over the feature map and keep only the maximum value.

    Stride is 1 here, so the output size is N - window + 1, the same rule
    as for the convolution.
    """
    n = feature_map.shape[0]
    out_size = n - window + 1
    output = np.zeros((out_size, out_size))

    for row in range(out_size):
        for col in range(out_size):
            patch = feature_map[row:row + window, col:col + window]
            output[row, col] = np.max(patch)
    return output


# Step 5: 2x2 max pooling on the convolution output.
pooled = max_pool2d(conv_out, window=2)
print("\nStep 5: 2x2 max pooling output, rounded to 2 decimals")
print(np.round(pooled, 2))
print(f"Pooling output size: {conv_out.shape[0]} - 2 + 1 = {pooled.shape[0]}")

## Part B: train a small CNN with PyTorch (`train_small_cnn.py`)

This part covers README Steps 6 to 9: load the digit images, build a small CNN, train it for 25 epochs, and measure on the test set. The images come from scikit-learn, so no download is needed.

**GPU (optional):** On Colab you can turn one on with Runtime > Change runtime type > GPU. CPU also works. The code below runs on the CPU as written and does not move the model or data to the GPU, so a GPU will not speed it up unless that code is changed.

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

SEED = 42
EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 0.001

torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)  # keeps small CPU runs predictable

# Step 1: Load the images.
digits = load_digits()
images = digits.images.astype(np.float32)  # shape (1797, 8, 8), values 0 to 16
labels = digits.target.astype(np.int64)
print(f"Loaded {len(images)} images of size {images.shape[1]}x{images.shape[2]}")

# Step 2: Normalize pixels to 0 to 1, and add a channel dimension.
# CNNs in PyTorch expect shape (batch, channels, height, width).
images = images / 16.0
images = images[:, np.newaxis, :, :]  # shape (1797, 1, 8, 8)

# Step 3: Split into training and test sets (80/20, same class mix in each).
x_train, x_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=SEED, stratify=labels
)
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)
print(f"Training images: {len(x_train)}, test images: {len(x_test)}")

# Step 4: Build a small CNN.
#   Conv -> ReLU -> Pool, twice, then Flatten -> Linear (the classifier).
#   8x8 input -> conv keeps 8x8 (padding=1) -> pool -> 4x4 -> conv -> pool -> 2x2
model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),        # 8x8 -> 4x4
    nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),        # 4x4 -> 2x2
    nn.Flatten(),                       # 16 channels x 2 x 2 = 64 values
    nn.Linear(16 * 2 * 2, 10),          # 10 outputs, one per digit
)
print("\nModel:")
print(model)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def accuracy(inputs, targets):
    """Return the fraction of predictions that match the true labels."""
    model.eval()
    with torch.no_grad():
        predictions = model(inputs).argmax(dim=1)
    return (predictions == targets).float().mean().item()


# Step 5: Training loop. One epoch = one pass over all training images.
print("\nTraining:")
for epoch in range(1, EPOCHS + 1):
    model.train()
    order = torch.randperm(len(x_train))
    running_loss = 0.0

    for start in range(0, len(x_train), BATCH_SIZE):
        batch_index = order[start:start + BATCH_SIZE]
        batch_x = x_train[batch_index]
        batch_y = y_train[batch_index]

        optimizer.zero_grad()                 # clear old gradients
        outputs = model(batch_x)              # forward propagation
        loss = loss_function(outputs, batch_y)
        loss.backward()                       # backward propagation
        optimizer.step()                      # update the weights

        running_loss += loss.item() * len(batch_index)

    average_loss = running_loss / len(x_train)
    train_acc = accuracy(x_train, y_train)
    print(f"Epoch {epoch:2d}/{EPOCHS}  loss {average_loss:.4f}  train accuracy {train_acc:.3f}")

# Step 6: Measure on the test set, which the model has never trained on.
test_acc = accuracy(x_test, y_test)
correct = round(test_acc * len(x_test))
print(f"\nTest accuracy: {test_acc:.3f} ({correct}/{len(x_test)} correct)")

## Checkpoints and what you learned

- After Step 1 to 2: A 5x5 matrix of pixel values, then the same grid with values between 0 and 1
- After Step 4: `Check passed: output is 3x3, from 25 values down to 9`
- After Step 5: A 2x2 matrix of values (the first row is about -2.10 and 2.06), and the line `Pooling output size: 3 - 2 + 1 = 2`
- After Step 6: `Loaded 1797 images of size 8x8` and `Training images: 1437, test images: 360`
- After Step 8: Twenty-five epoch lines. The loss goes down and the training accuracy goes up, to about 0.96
- After Step 9: `Test accuracy: 0.939 (338/360 correct)` in the reference run

Results can differ a little between machines. A test accuracy of 0.90 or higher counts as success for this exercise.

## Run locally instead?

See the README setup section.